In [ ]:
import numpy as np
import os, sys
import importlib.util

# Load the cross section modules
spec = importlib.util.spec_from_file_location(
    "fidXS_PTH_ggH",
    "/work/niharrin/devel/RCR_Matrix/fidXS/fidXS_PTH_ggH.py"
)
ggh_xs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ggh_xs)

spec = importlib.util.spec_from_file_location(
    "fidXS_PTH_VBFH",
    "/work/niharrin/devel/RCR_Matrix/fidXS/fidXS_PTH_VBFH.py"
)
vbf_xs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vbf_xs)

spec = importlib.util.spec_from_file_location(
    "fidXS_PTH_VH",
    "/work/niharrin/devel/RCR_Matrix/fidXS/fidXS_PTH_VH.py"
)
vh_xs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vh_xs)

spec = importlib.util.spec_from_file_location(
    "fidXS_PTH_ttH",
    "/work/niharrin/devel/RCR_Matrix/fidXS/fidXS_PTH_ttH.py"
)
tth_xs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tth_xs)

In [87]:
def format_bin_edge(edge: float) -> str:
    """
    Converts a numerical bin edge to the datacard string format.
    Example: 5.0 -> '5p0', 10000 -> '10000p0'
    """
    return str(float(edge)).replace('.', 'p')

def generate_nuisance_lines(
    datacard_path: str, 
    bins: list, 
    nuisances_data: dict, 
    output_path: str
):
    """
    Parses a combine datacard, finds the relevant bin/process columns, 
    and appends the formatted nuisance parameters.
    """
    
    # 1. Build a mapping from the string signature to the array index.
    # This prevents partial substring bugs (e.g., confusing 5p0 with 15p0).
    bin_map = {}
    for i in range(len(bins) - 1):
        low_str = format_bin_edge(bins[i])
        high_str = format_bin_edge(bins[i+1])
        # We look for the surrounding underscores to ensure an exact match
        signature = f"_{low_str}_{high_str}_"
        bin_map[signature] = i

    # 2. Read the existing datacard
    with open(datacard_path, 'r') as f:
        lines = f.readlines()

    bin_tokens = []
    process_tokens = []

    # 3. Locate the correct 'bin' and 'process' lines. 
    # The correct 'bin' line is usually right above 'process' and they share the same length.
    for i, line in enumerate(lines):
        if line.startswith('bin'):
            # Look ahead to find the associated 'process' line
            for j in range(1, 3):
                if i + j < len(lines) and lines[i + j].startswith('process'):
                    b_toks = line.strip().split()
                    p_toks = lines[i + j].strip().split()
                    
                    # Combine has a process *name* line and a process *ID* line (0, 1, -1). 
                    # We want the one with names, so we ensure the second element isn't a digit.
                    if len(b_toks) == len(p_toks) and not p_toks[1].lstrip('-').isdigit():
                        bin_tokens = b_toks
                        process_tokens = p_toks
                        break
            if bin_tokens:
                break

    if not bin_tokens or not process_tokens:
        raise ValueError("Could not find matching 'bin' and 'process' lines in the datacard.")

    # 4. Generate the new nuisance lines
    new_nuisance_lines = []
    
    for nuisance_name, (up_arr, dn_arr) in nuisances_data.items():
        # Start the line with the nuisance name and its type
        line_parts = [f"{nuisance_name:<30}", "lnN"]
        
        # Zip through the columns (skipping the first token which is 'bin' / 'process')
        for b_col, p_col in zip(bin_tokens[1:], process_tokens[1:]):
            if p_col == 'bkg_mass':
                line_parts.append("-")
                continue
            
            # Identify which interval this bin belongs to
            found_idx = -1
            for signature, idx in bin_map.items():
                if signature in b_col:
                    found_idx = idx
                    break
            
            if found_idx != -1:
                # Format to 3 decimal places to keep the datacard clean
                up_val = up_arr[found_idx]
                dn_val = dn_arr[found_idx]
                line_parts.append(f"{up_val:.3f}/{dn_val:.3f}")
            else:
                # If a signal process doesn't match our bins, default to '-'
                line_parts.append("-")
        
        # Join with consistent spacing and append to our new lines list
        new_nuisance_lines.append("   ".join(line_parts) + "\n")

    # 5. Write everything to the new output datacard
    with open(output_path, 'w') as f:
        f.writelines(lines)
        if not lines[-1].endswith('\n'):
            f.write('\n')
        # Add a visual separator for cleanliness
        f.write("--------------------------------------------------\n")
        f.writelines(new_nuisance_lines)
        
    print(f"Successfully added {len(nuisances_data)} nuisance parameters to {output_path}")

In [76]:
# fidXS
fidXS = {}

# List of variables to import
vars = ['fidXS', 'fidXS_scale_up', 'fidXS_scale_dn', 'fidXS_pdf_up', 'fidXS_pdf_dn', 'fidXS_alpha_up', 'fidXS_alpha_dn', 'Boundaries']

# Dynamically import the required module for ggH cross-section
bins = ggh_xs.Boundaries
bins_plot = np.array(bins)

# if "overflow" in current_config.keys(): bins_plot[-1] = 500
bins_plot[-1] = 10000 # We look at PTH
bins[-1] = 10000

# Calculate bin centers and widths
bins_c = (bins_plot[1:]+bins_plot[:-1])*0.5
bin_w = np.array([bins_plot[k+1]-bins_plot[k] for k in range(len(bins)-1)])
fidXS['ggh'] = np.array(ggh_xs.fidXS)
ggh_acc_norm = fidXS['ggh'] / bin_w

# Dynamically import the required module for xH cross-section
# xh_acc = __import__(current_config['xh_acc'], globals(), locals(), vars)
# xh_acc_norm = np.array(xh_acc.Acc) / bin_w

# Dynamically import the required module for VBF, VH, and ttH cross-section
fidXS['vbf'] = np.array(vbf_xs.fidXS)
vbf_acc_norm = fidXS['vbf'] / bin_w

fidXS['vh'] = np.array(vh_xs.fidXS)
vh_acc_norm = fidXS['vh'] / bin_w

fidXS['tth'] = np.array(tth_xs.fidXS)
tth_acc_norm = fidXS['tth'] / bin_w


# Theoretical uncertainty
sources_up = ["fidXS_scale_up", "fidXS_pdf_up", "fidXS_alpha_up"]

## [[fidXS_scale_up unc per bin], [fidXS_pdf_up unc per bin], [fidXS_alpha_up unc per bin]]
ggh_up = [(abs(np.array(getattr(ggh_xs, source)) - np.array(ggh_xs.fidXS))) / np.array(ggh_xs.fidXS) for source in sources_up]
vbf_up = [(abs(np.array(getattr(vbf_xs, source)) - np.array(vbf_xs.fidXS))) / np.array(vbf_xs.fidXS) for source in sources_up]
vh_up = [(abs(np.array(getattr(vh_xs, source)) - np.array(vh_xs.fidXS))) / np.array(vh_xs.fidXS) for source in sources_up]
tth_up = [(abs(np.array(getattr(tth_xs, source)) - np.array(tth_xs.fidXS))) / np.array(tth_xs.fidXS) for source in sources_up]

sources_dn = ["fidXS_scale_dn", "fidXS_pdf_dn", "fidXS_alpha_dn"]

ggh_dn = [(abs(np.array(getattr(ggh_xs, source)) - np.array(ggh_xs.fidXS))) / np.array(ggh_xs.fidXS) for source in sources_dn]
vbf_dn = [(abs(np.array(getattr(vbf_xs, source)) - np.array(vbf_xs.fidXS))) / np.array(vbf_xs.fidXS) for source in sources_dn]
vh_dn = [(abs(np.array(getattr(vh_xs, source)) - np.array(vh_xs.fidXS))) / np.array(vh_xs.fidXS) for source in sources_dn]
tth_dn = [(abs(np.array(getattr(tth_xs, source)) - np.array(tth_xs.fidXS))) / np.array(tth_xs.fidXS) for source in sources_dn]

ggh = np.array(ggh_up)
vbf = np.array(vbf_up)
vh = np.array(vh_up)
tth = np.array(tth_up)

## sum of the contributions for each production mode
## Linear sum as each source of uncertainty is fully correlated across the production modes
madgraph_up = np.sum([ggh_up,vbf_up,vh_up,tth_up], axis=0)
madgraph_dn = np.sum([ggh_dn,vbf_dn,vh_dn,tth_dn], axis=0)

In [77]:
np.array(ggh_xs.fidXS)

array([3.21783899, 6.32623678, 6.41838154, 5.71455977, 4.87754408,
       4.17624883, 3.58881896, 5.84540061, 5.96783699, 4.74666746,
       2.87391471, 1.7962164 , 1.196909  , 1.19259057, 0.73110078,
       0.69451504, 0.5780708 , 0.19049989, 0.12704297])

In [78]:
[(abs(np.array(getattr(ggh_xs, source)))) for source in sources_up]

[array([3.60996207, 7.07136174, 7.14785781, 6.31071305, 5.31201353,
        4.46056797, 3.73603637, 6.24069358, 6.54871248, 5.28278594,
        3.2410617 , 2.02901982, 1.3401989 , 1.35491569, 0.83378254,
        0.79513015, 0.66951433, 0.21769655, 0.14547503]),
 array([3.30020002, 6.48814581, 6.58200616, 5.85958396, 4.99961761,
        4.27831103, 3.67506054, 5.98113227, 6.10175132, 4.85133444,
        2.938146  , 1.83780838, 1.22551842, 1.22263454, 0.75010701,
        0.71387791, 0.59570028, 0.19707422, 0.13184014]),
 array([3.35119065, 6.58597704, 6.67823181, 5.93779965, 5.05622658,
        4.31258047, 3.68798182, 5.97271819, 6.06055875, 4.79728501,
        2.89679713, 1.80805714, 1.20457308, 1.19924614, 0.73500872,
        0.69826857, 0.58156225, 0.19171377, 0.12795536])]

In [79]:
madgraph_up

array([[0.2714764 , 0.28955244, 0.21616494, 0.28474055, 0.20640592,
        0.16335021, 0.15981084, 0.1696625 , 0.20176958, 0.20188623,
        0.2152485 , 0.21918913, 0.19702745, 0.22399276, 0.22755575,
        0.2244095 , 0.25822385, 0.24058305, 0.30096764],
       [0.12818586, 0.10784327, 0.10540224, 0.1052694 , 0.10776218,
        0.10566986, 0.1048396 , 0.09975636, 0.0981085 , 0.09761424,
        0.09982526, 0.09949745, 0.10406956, 0.10214937, 0.10532726,
        0.11170647, 0.12003933, 0.14274264, 0.16898955],
       [0.08328859, 0.08010057, 0.07767691, 0.07659992, 0.07320513,
        0.07342701, 0.06414097, 0.05824722, 0.05327961, 0.04883067,
        0.04705279, 0.0451977 , 0.04394232, 0.04386409, 0.04395124,
        0.04256884, 0.04577916, 0.04733772, 0.03418858]])

In [80]:
# Theoretical uncertainty
sources_up = ["fidXS_scale_up", "fidXS_pdf_up", "fidXS_alpha_up"]

## [[fidXS_scale_up unc per bin], [fidXS_pdf_up unc per bin], [fidXS_alpha_up unc per bin]]
ggh_up = [(abs(np.array(getattr(ggh_xs, source)) - np.array(ggh_xs.fidXS))) for source in sources_up]
vbf_up = [(abs(np.array(getattr(vbf_xs, source)) - np.array(vbf_xs.fidXS))) for source in sources_up]
vh_up = [(abs(np.array(getattr(vh_xs, source)) - np.array(vh_xs.fidXS))) for source in sources_up]
tth_up = [(abs(np.array(getattr(tth_xs, source)) - np.array(tth_xs.fidXS))) for source in sources_up]

sources_dn = ["fidXS_scale_dn", "fidXS_pdf_dn", "fidXS_alpha_dn"]

ggh_dn = [(abs(np.array(getattr(ggh_xs, source)) - np.array(ggh_xs.fidXS))) for source in sources_dn]
vbf_dn = [(abs(np.array(getattr(vbf_xs, source)) - np.array(vbf_xs.fidXS))) for source in sources_dn]
vh_dn = [(abs(np.array(getattr(vh_xs, source)) - np.array(vh_xs.fidXS))) for source in sources_dn]
tth_dn = [(abs(np.array(getattr(tth_xs, source)) - np.array(tth_xs.fidXS))) for source in sources_dn]

madgraph_up = np.sum([ggh_up,vbf_up,vh_up,tth_up], axis=0)
madgraph_dn = np.sum([ggh_dn,vbf_dn,vh_dn,tth_dn], axis=0)

theor_up = madgraph_up / (fidXS['ggh']+fidXS['vbf']+fidXS['vh']+fidXS['tth'])
theor_dn = madgraph_dn / (fidXS['ggh']+fidXS['vbf']+fidXS['vh']+fidXS['tth'])

In [81]:
dc_up = (1 + theor_up)
dc_dn = (1 - theor_dn)

In [82]:
CMS_hgg_scale_up = dc_up[0]
CMS_hgg_scale_dn = dc_dn[0]

CMS_hgg_pdfWeight_up = dc_up[1]
CMS_hgg_pdfWeight_dn = dc_dn[1]

CMS_hgg_AlphaS_up = dc_up[2]
CMS_hgg_AlphaS_dn = dc_dn[2]

In [84]:
bins

[0.0,
 5.0,
 10.0,
 15.0,
 20.0,
 25.0,
 30.0,
 35.0,
 45.0,
 60.0,
 80.0,
 100.0,
 120.0,
 140.0,
 170.0,
 200.0,
 250.0,
 350.0,
 450.0,
 10000]

In [88]:
# Map the root nuisance name to the (up, down) tuple
nuisances = {
    "CMS_hgg_scale": (CMS_hgg_scale_up, CMS_hgg_scale_dn),
    "CMS_hgg_pdfWeight": (CMS_hgg_pdfWeight_up, CMS_hgg_pdfWeight_dn),
    "CMS_hgg_AlphaS": (CMS_hgg_AlphaS_up, CMS_hgg_AlphaS_dn),
}

# Run the generator
generate_nuisance_lines(
    datacard_path='/work/niharrin/devel/RCR_Matrix/theoretical_uncertainties/datacards/Datacard_PTH_2022.txt', 
    bins=bins, 
    nuisances_data=nuisances, 
    output_path='/work/niharrin/devel/RCR_Matrix/theoretical_uncertainties/datacards/datacard_with_nuisances.txt'
)

Successfully added 3 nuisance parameters to /work/niharrin/devel/RCR_Matrix/theoretical_uncertainties/datacards/datacard_with_nuisances.txt
